**LangChain Expression Language**  
<small>LCEL connects AI building blocks like prompts, models, data retrievers and parsers by using a “pipe” symbol (|) to ensure smoothly  information flow from one part to another.  
Instead of writing complicated code, we just stack these blocks in the order we need and LCEL makes sure each step passes its output to the next.  
Instead of using:   
`chain = LLMChain(llm=llm, prompt=prompt)`   

we use:  
`chain = prompt | llm | parser`</nsmall>

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_anthropic import ChatAnthropic
import os
# LLM
llm = ChatAnthropic(
    model="claude-3-5-sonnet-20241022",
    temperature=0.7,
    anthropic_api_key=os.getenv("ANTHROPIC_API_KEY")
)
# Output parser
parser = StrOutputParser()

**Basic LCEL Chain**  
<small>Simple Prompt → LLM → Output Parser</nsmall>

In [ ]:
# Prompt
prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple terms."
)
# LCEL Chain
chain = prompt | llm | parser
result = chain.invoke({
    "topic": "LangChain agents"
})
print(result)

<small>For the expression:  
> prompt | llm | parser

LCEL internally creates the following pipeline:  
> Dict input --> PromptTemplate --> ChatModel --> OutputParser</nsmall>

**Multi-Step AI Workflow**  
<small>Example:  
Generate a summary  
Translate summary to French  
Format output</nsmall>

In [ ]:
# Prompt templates
summary_prompt = ChatPromptTemplate.from_template(
    "Summarize this text:\n\n{text}"
)
translation_prompt = ChatPromptTemplate.from_template(
    "Translate this into French:\n\n{text}"
)
# Individual chains
summary_chain = summary_prompt | llm | parser
translation_chain = translation_prompt | llm | parser
# Chain composition
full_chain = summary_chain | translation_chain
result = full_chain.invoke({
    "text": "LangChain helps developers build LLM applications."
})
print(result)

**Parallel Execution**  
<small>LCEL supports parallel branches</nsmall>

In [ ]:
from langchain_core.runnables import RunnableParallel
analysis = RunnableParallel(
    summary=summary_chain,
    translation=translation_chain,
)
result = analysis.invoke({
    "text": "AI agents can use tools."
})
print(result)

**RunnableLambda for Custom Logic**  
<small>we can insert Python functions into chains.</nsmall>

In [ ]:
from langchain_core.runnables import RunnableLambda
def clean_text(text: str):
    return {"text": text.strip().lower()}
cleaner = RunnableLambda(clean_text)
chain = cleaner | summary_chain
result = chain.invoke("   HELLO WORLD   ")
print(result)

**Branching and Routing**  
<small>Routing requests dynamically.</nsmall>

In [ ]:
from langchain_core.runnables import RunnableBranch
prompt_math = ChatPromptTemplate.from_template(
    "Solve this math question:\n\n{query}"
)
prompt_code = ChatPromptTemplate.from_template(
    "Explain this programming question:\n\n{query}"
)
math_chain = prompt_math | llm | parser
code_chain = prompt_code | llm | parser
router = RunnableBranch(
    (lambda x: "math" in x["query"].lower(), math_chain),
    code_chain
)
result = router.invoke({
    "query": "math problem"
})
print(result)

**Streaming**

In [ ]:
prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple terms."
)
chain = prompt | llm | parser
for chunk in chain.stream({
    "topic": "AI agents"
}):
    print(chunk, end="", flush=True)

**Async "Invoke + Branch" execution**

In [ ]:
async def main():
    # Single async call
    result = await chain.ainvoke({
        "topic": "LangChain"
    })
    print("Single Result:")
    print(result)
    # Parallel async batch
    results = await chain.abatch([
        {"topic": "AI"},
        {"topic": "Agents"},
    ])
    print("\nBatch Results:")
    for r in results:
        print(r)
await main()

**Memory Management**  
<small>Old approach:   
`ConversationBufferMemory`  

Problems:
- Hard to scale  
- Hidden state  
- Limited persistence  
- Not composable  

Modern LCEL memory:  
- Explicit  
- Runnable-based  
- Easier debugging  
- Better production support  

Modern LangChain prefers:  
- RunnableWithMessageHistory  
- LangGraph memory  
- External stores (Redis, Postgres)</nsmall>

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])
# Chain
chain = prompt | llm
# Store chat histories
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Memory-enabled chain
conversation_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)
# First message
response = conversation_chain.invoke(
    {"input": "My name is Akash"},
    config={"configurable": {"session_id": "user1"}}
)
# Follow-up message
response = conversation_chain.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "user1"}}
)
print(response.content)